# A. Gradient Check

Numerical gradient verification using central difference method.

- **A1**: Implement gradient check
- **A2**: Compare weight/bias gradient per layer (relative error < 1e-5)
- **A3**: Verify softmax + cross-entropy combined gradient

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
from network import Network, NetworkConfig

## A1 & A2: Numerical Gradient Check per Layer

In [ ]:
def numerical_gradient(network, x, y, param_type, layer_idx, epsilon=1e-5):
    """Compute numerical gradient using central difference."""
    params = network.weights[layer_idx] if param_type == 'weight' else network.biases[layer_idx]
    num_grad = np.zeros_like(params)
    it = np.nditer(params, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old_val = float(params[idx])

        params[idx] = old_val + epsilon
        loss_plus = network.loss(y, network.forward(x))

        params[idx] = old_val - epsilon
        loss_minus = network.loss(y, network.forward(x))

        params[idx] = old_val
        num_grad[idx] = (loss_plus - loss_minus) / (2 * epsilon)
        it.iternext()
    return num_grad


def relative_error(numerical, analytical):
    """Compute max relative error."""
    denom = np.maximum(np.abs(numerical) + np.abs(analytical), 1e-8)
    return np.max(np.abs(numerical - analytical) / denom)

### Test: ReLU + Softmax + Cross-Entropy

In [ ]:
np.random.seed(42)

config = NetworkConfig(
    layers=[4, 5, 3, 2],
    activation='relu',
    loss='cross_entropy',
    output_activation='softmax',
    weights_initializer='heUniform'
)
net = Network(config)

x = np.random.randn(5, 4)
y = np.array([[1,0],[0,1],[1,0],[0,1],[1,0]])

net.forward(x)
nabla_w, nabla_b = net.backward(y)

print('=' * 60)
print('Gradient Check: ReLU + Softmax + Cross-Entropy')
print('=' * 60)

all_passed = True
for i in range(len(net.weights)):
    num_gw = numerical_gradient(net, x, y, 'weight', i)
    err_w = relative_error(num_gw, nabla_w[i])
    ok_w = err_w < 1e-5
    if not ok_w: all_passed = False
    print(f'Layer {i} Weight | Rel Error: {err_w:.2e} | {"PASS" if ok_w else "FAIL"}')

    num_gb = numerical_gradient(net, x, y, 'bias', i)
    err_b = relative_error(num_gb, nabla_b[i])
    ok_b = err_b < 1e-5
    if not ok_b: all_passed = False
    print(f'Layer {i} Bias   | Rel Error: {err_b:.2e} | {"PASS" if ok_b else "FAIL"}')

print('=' * 60)
print(f'Overall: {"ALL PASSED" if all_passed else "SOME FAILED"}')

## A3: Softmax + CE with Sigmoid Hidden Layers

In [ ]:
np.random.seed(123)

config_sig = NetworkConfig(
    layers=[4, 3, 2],
    activation='sigmoid',
    loss='cross_entropy',
    output_activation='softmax',
    weights_initializer='xavierUniform'
)
net_sig = Network(config_sig)

x2 = np.random.randn(3, 4) * 0.5
y2 = np.array([[1,0],[0,1],[1,0]])

net_sig.forward(x2)
nabla_w2, nabla_b2 = net_sig.backward(y2)

print('=' * 60)
print('Gradient Check: Sigmoid + Softmax + Cross-Entropy')
print('=' * 60)

all_passed = True
for i in range(len(net_sig.weights)):
    num_gw = numerical_gradient(net_sig, x2, y2, 'weight', i)
    err_w = relative_error(num_gw, nabla_w2[i])
    ok_w = err_w < 1e-5
    if not ok_w: all_passed = False
    print(f'Layer {i} Weight | Rel Error: {err_w:.2e} | {"PASS" if ok_w else "FAIL"}')

    num_gb = numerical_gradient(net_sig, x2, y2, 'bias', i)
    err_b = relative_error(num_gb, nabla_b2[i])
    ok_b = err_b < 1e-5
    if not ok_b: all_passed = False
    print(f'Layer {i} Bias   | Rel Error: {err_b:.2e} | {"PASS" if ok_b else "FAIL"}')

print('=' * 60)
print(f'Overall: {"ALL PASSED" if all_passed else "SOME FAILED"}')